In [105]:
import h5py
import numpy as np
import random
import json
import sympy as sp

def parse_tokens(tokens, pos=0):
    """
    Recursively parse a list of tokens (starting at pos) into a Sympy expression.
    This parser expects tokens of the form:
      - For function calls: function_name, "(", arg1, ",", arg2, ..., ")"
      - For atomic tokens: anything not followed by "(" is taken as a symbol (or number).
    This simple version maps '+' to sp.Add, '*' to sp.Mul, '^' to sp.Pow,
    and 'DP' to a custom DP function.
    """
    # If no more tokens, return None.
    if pos >= len(tokens):
        raise ValueError("Ran out of tokens while parsing.")
    
    token = tokens[pos]
    
    # If the next token is "(", it's a function call.
    if pos + 1 < len(tokens) and tokens[pos + 1] == "(":
        func_token = token
        pos += 2  # skip token and "("
        args = []
        while pos < len(tokens) and tokens[pos] != ")":
            arg, pos = parse_tokens(tokens, pos)
            args.append(arg)
            if pos < len(tokens) and tokens[pos] == ",":
                pos += 1
        if pos >= len(tokens) or tokens[pos] != ")":
            raise ValueError("Expected ')'")
        pos += 1  # skip the ")"
        # Map common operator tokens to sympy functions.
        if func_token == "+":
            return sp.Add(*args), pos
        elif func_token == "*":
            return sp.Mul(*args), pos
        elif func_token == "^":
            if len(args) != 2:
                raise ValueError("Operator '^' requires exactly 2 arguments")
            return sp.Pow(args[0], args[1]), pos
        elif func_token == "DP":
            # For our custom DP function, assume it is defined.
            return DP(*args), pos
        else:
            # For any other function, use a generic sympy Function.
            f = sp.Function(func_token)
            return f(*args), pos
    else:
        # Atomic token: try to interpret as number, otherwise as symbol.
        try:
            # If it contains a dot, convert to float.
            if '.' in token:
                atom = sp.Float(token)
            else:
                atom = sp.Integer(token)
        except ValueError:
            atom = sp.Symbol(token)
        return atom, pos + 1

def detokenize_expr(tokens):
    """
    Convert a token list back into a Sympy expression using parse_tokens.
    """
    expr, pos = parse_tokens(tokens, 0)
    if pos != len(tokens):
        raise ValueError("Extra tokens remaining after parsing")
    return expr

# Define our custom DP function (for completeness).
class DP(sp.Function):
    nargs = 2
    @classmethod
    def eval(cls, a, b):
        return None  # No automatic simplification.
    def _sympystr__(self, printer):
        a, b = self.args
        return f"DP({printer._print(a)}, {printer._print(b)})"

# --- Function to load random entries from the HDF5 file ---
def load_random_entries(filename, num_entries=5, dataset_name="simple"):
    """
    Opens the HDF5 file, loads the dataset (either 'simple' or 'scrambled'),
    retrieves a few random entries, and converts each back into a Sympy expression.
    
    Returns a list of (tokens, expression) tuples.
    """
    results = []
    with h5py.File(filename, "r") as f:
        # Load the vocabulary from the file attributes (stored as JSON).
        vocab_json = f.attrs.get("vocab")
        if vocab_json is None:
            raise ValueError("No vocabulary found in file attributes.")
        vocab = json.loads(vocab_json)
        # Build the inverse vocabulary (index to token).
        inv_vocab = {int(idx): token for token, idx in vocab.items()}
        
        # Load the dataset.
        data = f[dataset_name]
        total = data.shape[0]
        # Choose num_entries random indices.
        indices = random.sample(range(total), num_entries)
        
        for idx in indices:
            # Each entry is a variable-length array of int32.
            token_indices = data[idx]
            # Convert indices to tokens using the inverse vocabulary.
            tokens = [inv_vocab.get(int(i), "UNK") for i in token_indices]
            # Use our detokenizer to rebuild the sympy expression.
            expr = detokenize_expr(tokens)
            results.append((tokens, expr))
    return results

# --- Demo ---
if __name__ == "__main__":
    # Replace "amplitude_5_particle_dataset.hdf5" with your filename.
    filename = "amplitude_3_particle_dataset.hdf5"
    # For demonstration, we load random entries from the "simple" dataset.
    entries = load_random_entries(filename, num_entries=5, dataset_name="simple")
    for i, (tokens, expr) in enumerate(entries):
        print(f"Entry {i}:")
       # print("Tokens:")
       # print(tokens)
       # print("Reconstructed Expression:")
        print(expr)


Entry 0:
DP(e1, e2)*DP(e3, p2) + DP(e1, p3)*DP(e3, e2) + DP(e2, p1)*DP(e3, e1)
Entry 1:
DP(e1, e2)*DP(e3, p2)
Entry 2:
DP(e1, e2)*DP(e3, p2) + DP(e1, p2)*DP(e2, e3)
Entry 3:
DP(e1, e3)*DP(e2, p1) + DP(e1, e3)*DP(e2, p3) + DP(e2, p3)*DP(e3, e1)
Entry 4:
DP(e1, p2)*DP(e3, e2)
